# Practical Guide to Boosting Algorithms in Machine Learning

Complete maintained companion: AdaBoost classification/regression, residual-fitting trees, gradient boosting, validation-stage selection and XGBoost classification/regression. Python 3.11+, CPU, no API keys/external data. Run top to bottom. macOS XGBoost requires an OpenMP runtime (for example the official Homebrew libomp package) if not already installed. Source snapshots remain unchanged in the appendix.

Maintenance: missing AdaBoost fit supplied (500 stumps, learning_rate=1); sklearn 1.7 uses discrete SAMME, not removed SAMME.R. Modern seaborn keyword arguments and MSE API, explicit separate regression data, malformed subplot and mistaken plotting arguments fixed. Both MSE and RMSE printed separately. A validation minimum over 120 fitted stages is selection, not an untouched final test. Source claims about parallel trees, overfitting immunity and parameter meanings have private findings in the website review; this notebook does not assert them.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                'numpy==2.3.3', 'scipy==1.16.2', 'scikit-learn==1.7.2',
                'matplotlib==3.10.9', 'joblib==1.5.2', 'threadpoolctl==3.6.0',
                'seaborn==0.13.2', 'pandas==2.3.3', 'xgboost==3.0.5'], check=True)
import json
import tempfile
from pathlib import Path
import numpy as np
import sklearn
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, root_mean_squared_error, accuracy_score
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits
%matplotlib inline
thread_limits = threadpool_limits(limits=1)
np.set_printoptions(precision=8, linewidth=100)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| scikit-learn:', sklearn.__version__)
print('CPU, one numerical thread; original seeded synthetic datasets; no keys or external data.')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons
import seaborn as sns 

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

plt.figure(figsize=(10,8))
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette="deep")
plt.title('The moon dataset', fontsize=18)
plt.xlabel(r"X1", fontsize=18)
plt.ylabel(r"X2", fontsize=18, rotation=0)
plt.show()
X_moons, y_moons = X.copy(), y.copy()
Xm_train, Xm_test, ym_train, ym_test = X_train.copy(), X_test.copy(), y_train.copy(), y_test.copy()
print('Moons:', X.shape, '| Train/test:', len(ym_train), len(ym_test))

In [ ]:
ada_clf = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                             n_estimators=500, learning_rate=1.0, random_state=42)
ada_clf.fit(Xm_train, ym_train)
print('AdaBoost fitted stumps:', len(ada_clf.estimators_))
print('AdaBoost train accuracy:', round(ada_clf.score(Xm_train, ym_train), 6))
print('AdaBoost test accuracy:', round(ada_clf.score(Xm_test, ym_test), 6))

In [ ]:
from matplotlib.colors import ListedColormap

def plot_decision_boundary(clf, X, y, axes=[-1.5, 2.45, -1, 1.5], alpha=0.5, contour=True):
    x1s = np.linspace(axes[0], axes[1], 100)
    x2s = np.linspace(axes[2], axes[3], 100)
    x1, x2 = np.meshgrid(x1s, x2s)
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_new).reshape(x1.shape)
    custom_cmap = ListedColormap(['#fafab0','#9898ff','#a0faa0'])
    plt.contourf(x1, x2, y_pred, alpha=0.3, cmap=custom_cmap)
    if contour:
        custom_cmap2 = ListedColormap(['#7d7d58','#4c4c7f','#507d50'])
        plt.contour(x1, x2, y_pred, cmap=custom_cmap2, alpha=0.8)
    plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo", alpha=alpha)
    plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs", alpha=alpha)
    plt.axis(axes)
    plt.xlabel(r"$x_1$", fontsize=18)
    plt.ylabel(r"$x_2$", fontsize=18, rotation=0)
    

plt.figure(figsize=(8, 6))
plot_decision_boundary(ada_clf, X, y)
plt.title('AdaBoost decision boundaries')
plt.show()
print('AdaBoost decision regions rendered.')

In [ ]:
np.random.seed(42)

X = np.random.rand(100, 1) - 0.5
y = 3*X[:, 0]**2 + 0.05 * np.random.randn(100)
plt.plot(X,y, '.')
plt.title('Quadratic Data')
plt.xlabel('X')
plt.ylabel('y')
plt.show()
X_quad, y_quad = X.copy(), y.copy()
print('Quadratic data:', X.shape, '| First targets:', y[:3])

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.datasets import make_regression

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=49)
regr = AdaBoostRegressor(random_state=0, n_estimators=100)
regr.fit(X_train, y_train)
y_pred = regr.predict(X_val)
print('AdaBoost validation MSE:', round(mean_squared_error(y_val, y_pred), 8))
print('AdaBoost validation RMSE:', round(root_mean_squared_error(y_val, y_pred), 8))
Xq_train, Xq_val, yq_train, yq_val = X_train.copy(), X_val.copy(), y_train.copy(), y_val.copy()

In [ ]:
np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3*X[:, 0]**2 + 0.05 * np.random.randn(100)
plt.plot(X,y, '.')
plt.title('Quadratic Data')
plt.xlabel('X')
plt.ylabel('y')
plt.show()
X_quad, y_quad = X.copy(), y.copy()
print('Quadratic data:', X.shape, '| First targets:', y[:3])

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg1 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg1.fit(X, y)
print('First tree training MSE:', round(mean_squared_error(y, tree_reg1.predict(X)), 8))

In [ ]:
y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg2.fit(X, y2)
print('Residual MSE after two trees:', round(np.mean((y2-tree_reg2.predict(X))**2), 8))

In [ ]:
y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg3.fit(X, y3)
print('Residual MSE after three trees:', round(np.mean((y3-tree_reg3.predict(X))**2), 8))

In [ ]:
X_new = np.array([[0.8]])
y_pred = sum(tree.predict(X_new) for tree in (tree_reg1, tree_reg2, tree_reg3))
print('Three-tree prediction at x=0.8:', y_pred)

In [ ]:
def plot_predictions(regressors, X, y, axes, label=None, style="r-", data_style="b.", data_label=None):
    x1 = np.linspace(axes[0], axes[1], 500)
    y_pred = sum(regressor.predict(x1.reshape(-1, 1)) for regressor in regressors)
    plt.plot(X[:, 0], y, data_style, label=data_label)
    plt.plot(x1, y_pred, style, linewidth=2, label=label)
    if label or data_label:
        plt.legend(loc="upper center", fontsize=16)
    plt.axis(axes)
print('Additive prediction plotting helper ready.')

In [ ]:
plt.figure(figsize=(11,11))

plt.subplot(321)
plot_predictions([tree_reg1], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h_1(x_1)$", style="g-", data_label="Training set")
plt.ylabel("$y$", fontsize=16, rotation=0)
plt.title("Residuals and tree predictions", fontsize=16)

plt.subplot(322)
plot_predictions([tree_reg1], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1)$", data_label="Training set")
plt.ylabel("$y$", fontsize=16, rotation=0)
plt.title("Ensemble predictions", fontsize=16)

plt.subplot(323)
plot_predictions([tree_reg2], X, y2, axes=[-0.5, 0.5, -0.5, 0.5], label="$h_2(x_1)$", style="g-", data_style="k+", data_label="Residuals")
plt.ylabel("$y - h_1(x_1)$", fontsize=16)

plt.subplot(324)
plot_predictions([tree_reg1, tree_reg2], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1) + h_2(x_1)$")
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.subplot(325)
plot_predictions([tree_reg3], X, y3, axes=[-0.5, 0.5, -0.5, 0.5], label="$h_3(x_1)$", style="g-", data_style="k+")
plt.ylabel("$y - h_1(x_1) - h_2(x_1)$", fontsize=16)
plt.xlabel("$x_1$", fontsize=16)

plt.subplot(326)
plot_predictions([tree_reg1, tree_reg2, tree_reg3], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1) + h_2(x_1) + h_3(x_1)$")
plt.xlabel("$x_1$", fontsize=16)
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.tight_layout()
plt.show()
print('All six residual/ensemble panels rendered.')

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

gbclf = GradientBoostingClassifier(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbclf.fit(X_train, y_train)
y_pred = gbclf.predict(X_test)
print('Test accuarcy using Gradient Boosting Classifier', accuracy_score(y_test, y_pred))
print('Gradient boosting train accuracy:', round(gbclf.score(X_train, y_train), 6))

In [ ]:
plt.figure(figsize=(8, 6))
plot_decision_boundary(gbclf, X_moons, y_moons)
plt.title('Gradient boosting classifier decision boundaries')
plt.show()
print('Gradient-boosting decision regions rendered.')

In [ ]:
X, y = X_quad.copy(), y_quad.copy()
from sklearn.ensemble import GradientBoostingRegressor

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbrt.fit(X, y)
print('Three-tree GBRT training MSE:', round(mean_squared_error(y, gbrt.predict(X)), 8))

In [ ]:
gbrt_slow = GradientBoostingRegressor(max_depth=2, n_estimators=200, learning_rate=0.1, random_state=42)
gbrt_slow.fit(X, y)
print('Two-hundred-tree GBRT training MSE:', round(mean_squared_error(y, gbrt_slow.predict(X)), 8))

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(10,4), sharey=True)

plt.sca(axes[0])
plot_predictions([gbrt], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="Ensemble predictions")
plt.title("learning_rate={}, n_estimators={}".format(gbrt.learning_rate, gbrt.n_estimators), fontsize=14)
plt.xlabel("$x_1$", fontsize=16)
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.sca(axes[1])
plot_predictions([gbrt_slow], X, y, axes=[-0.5, 0.5, -0.1, 0.8])
plt.title("learning_rate={}, n_estimators={}".format(gbrt_slow.learning_rate, gbrt_slow.n_estimators), fontsize=14)
plt.xlabel("$x_1$", fontsize=16)

plt.tight_layout()
plt.show()
print('Both shrinkage comparison plots rendered.')

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=49)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120, random_state=42)
gbrt.fit(X_train, y_train)

errors = [mean_squared_error(y_val, y_pred)
          for y_pred in gbrt.staged_predict(X_val)]
bst_n_estimators = np.argmin(errors) + 1

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=int(bst_n_estimators), random_state=42)
gbrt_best.fit(X_train, y_train)
print('Selected trees:', int(bst_n_estimators))
print('Minimum validation MSE:', round(float(np.min(errors)), 8))
print('Selected-model validation RMSE:', round(root_mean_squared_error(y_val, gbrt_best.predict(X_val)), 8))

In [ ]:
min_error = np.min(errors)
plt.figure(figsize=(15, 8))

plt.subplot(121)
plt.plot(np.arange(1, len(errors) + 1), errors, "b.-")
plt.plot([bst_n_estimators, bst_n_estimators], [0, min_error], "k--")
plt.plot([0, 120], [min_error, min_error], "k--")
plt.plot(bst_n_estimators, min_error, "ko")
plt.text(bst_n_estimators, min_error*1.2, "Minimum", ha="center", fontsize=14)
plt.xlim(0, 120)
plt.ylim(0, max(errors) * 1.1)
plt.xlabel("Number of trees")
plt.ylabel("Error", fontsize=16)
plt.title("Validation error", fontsize=14)

plt.subplot(122)
plot_predictions([gbrt_best], X, y, axes=[-0.5, 0.5, -0.1, 0.8])
plt.title("Best model (%d trees)" % bst_n_estimators, fontsize=14)
plt.ylabel("$y$", fontsize=16, rotation=0)
plt.xlabel("$x_1$", fontsize=16)

plt.tight_layout()
plt.show()
print('Validation selection curve and best-fit plot rendered.')

In [ ]:
# XGBoost was installed in the pinned setup cell.
import xgboost as xgb
print('XGBoost:', xgb.__version__)

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

# create the moon dataset
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


xgb_classifier = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3,
                                   objective='binary:logistic', base_score=0.5,
                                   tree_method='hist', device='cpu', n_jobs=1, random_state=42)
xgb_classifier.fit(X_train,y_train)

predictions = xgb_classifier.predict(X_test)
print("classifcation accuracy of XGBoost :",accuracy_score(y_test,predictions))
print('XGBoost train accuracy:', round(xgb_classifier.score(X_train, y_train), 6))

In [ ]:
plt.figure(figsize=(8, 6))
plot_decision_boundary(xgb_classifier, X_train, y_train)
plt.title('XGBoost decision boundaries')
plt.show()
print('XGBoost decision regions rendered.')

In [ ]:
# Regression counterpart promised by the original introduction.
xgb_regressor = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.3,
                                 objective='reg:squarederror', base_score=0.5,
                                 tree_method='hist', device='cpu', n_jobs=1, random_state=42)
xgb_regressor.fit(Xq_train, yq_train)
quad_predictions = xgb_regressor.predict(Xq_val)
print('XGBoost validation MSE:', round(mean_squared_error(yq_val, quad_predictions), 8))
print('XGBoost validation RMSE:', round(root_mean_squared_error(yq_val, quad_predictions), 8))
plt.figure(figsize=(8, 5))
plot_predictions([xgb_regressor], X_quad, y_quad, [-0.5, 0.5, -0.1, 0.8], label='XGBoost regression')
plt.xlabel('X')
plt.ylabel('y')
plt.tight_layout()
plt.show()

In [ ]:
from numpy.testing import assert_allclose, assert_array_equal
from sklearn.base import clone
assert X_moons.shape == (500, 2) and X_quad.shape == (100, 1)
assert len(Xm_train) == 375 and len(Xm_test) == 125
assert len(Xq_train) == 75 and len(Xq_val) == 25
assert_allclose(X_quad, np.random.RandomState(42).rand(100, 1) - 0.5)
# Direct traversal of sklearn trees, independent of estimator.predict.
def tree_value(estimator, inputs):
    t = estimator.tree_
    result = []
    for row in np.asarray(inputs, dtype=np.float32):
        node = 0
        while t.children_left[node] != t.children_right[node]:
            node = t.children_left[node] if row[t.feature[node]] <= t.threshold[node] else t.children_right[node]
        result.append(t.value[node].ravel())
    return np.array(result)
def regression_tree(estimator, inputs):
    return tree_value(estimator, inputs)[:, 0]
assert_allclose(y2, y_quad-regression_tree(tree_reg1, X_quad))
assert_allclose(y3, y2-regression_tree(tree_reg2, X_quad))
assert_allclose(y_pred if np.shape(y_pred)==(1,) else sum(t.predict([[0.8]]) for t in [tree_reg1, tree_reg2, tree_reg3]),
                sum(regression_tree(t, [[0.8]]) for t in [tree_reg1, tree_reg2, tree_reg3]))
# Reconstruct SAMME weighted votes and original training-weight recurrence.
weights = np.full(len(ym_train), 1/len(ym_train))
votes = np.zeros(len(ym_test))
for stage, (tree, alpha) in enumerate(zip(ada_clf.estimators_, ada_clf.estimator_weights_)):
    train_labels = np.argmax(tree_value(tree, Xm_train), axis=1)
    wrong = train_labels != ym_train
    error = float(weights @ wrong)
    assert_allclose(error, ada_clf.estimator_errors_[stage], atol=1e-12)
    assert_allclose(alpha, np.log((1-error)/error), rtol=1e-10, atol=1e-10)
    weights *= np.exp(alpha * wrong)
    weights /= weights.sum()
    test_labels = np.argmax(tree_value(tree, Xm_test), axis=1)
    votes += alpha * (2*test_labels-1)
assert_array_equal((votes>0).astype(int), ada_clf.predict(Xm_test))
# AdaBoost.R2 uses a weighted median, not an arithmetic mean.
reg_preds = np.column_stack([regression_tree(t, Xq_val) for t in regr.estimators_])
sorter = np.argsort(reg_preds, axis=1)
cum = np.cumsum(regr.estimator_weights_[:len(regr.estimators_)][sorter], axis=1)
median_pos = np.argmax(cum >= 0.5*cum[:, -1, None], axis=1)
manual_median = reg_preds[np.arange(len(Xq_val)), sorter[np.arange(len(Xq_val)), median_pos]]
assert_allclose(manual_median, regr.predict(Xq_val), atol=1e-12)
# Reconstruct every staged GBRT validation MSE from tree leaves + initial mean.
manual = np.full(len(Xq_val), float(gbrt.init_.constant_[0, 0]))
manual_errors=[]
for stage, tree in enumerate(gbrt.estimators_[:, 0]):
    manual += gbrt.learning_rate * regression_tree(tree, Xq_val)
    assert_allclose(manual, list(gbrt.staged_predict(Xq_val))[stage], atol=1e-12)
    manual_errors.append(float(np.mean((yq_val-manual)**2)))
assert_allclose(manual_errors, errors, atol=1e-12)
assert bst_n_estimators == int(np.argmin(manual_errors))+1
assert_allclose(gbrt_best.predict(Xq_val), list(gbrt.staged_predict(Xq_val))[bst_n_estimators-1], atol=1e-12)
# XGBoost JSON-tree oracle; covers every classifier/regressor tree and row.
def xgb_sum(model, inputs):
    def leaf(node, row):
        if 'leaf' in node:
            return node['leaf']
        value = row[int(node['split'][1:])]
        child_id = node['missing'] if np.isnan(value) else (node['yes'] if value < node['split_condition'] else node['no'])
        return leaf(next(c for c in node['children'] if c['nodeid']==child_id), row)
    trees=[json.loads(t) for t in model.get_booster().get_dump(dump_format='json')]
    return np.array([sum(leaf(t, row) for t in trees) for row in np.asarray(inputs, dtype=np.float32)])
raw_margin=xgb_sum(xgb_classifier, X_moons) # logit(base_score=0.5) is zero.
assert_allclose(raw_margin, xgb_classifier.predict(X_moons, output_margin=True), rtol=2e-6, atol=1e-5)
assert_allclose(1/(1+np.exp(-raw_margin)), xgb_classifier.predict_proba(X_moons)[:,1], atol=2e-6)
assert_array_equal((raw_margin>0).astype(int), xgb_classifier.predict(X_moons))
assert_allclose(0.5+xgb_sum(xgb_regressor, X_quad), xgb_regressor.predict(X_quad), atol=2e-6)
for model, inputs in [(xgb_classifier,X_moons),(xgb_regressor,X_quad),(regr,Xq_val),(gbrt_best,Xq_val)]:
    assert_allclose(model.predict(inputs[::-1])[::-1], model.predict(inputs))
with tempfile.TemporaryDirectory(prefix='tdb-boosting-') as folder:
    for name, model, inputs in [('classifier',xgb_classifier,X_moons),('regressor',xgb_regressor,X_quad)]:
        path=Path(folder)/(name+'.json');model.save_model(path)
        restored=type(model)();restored.load_model(path)
        assert_allclose(restored.predict(inputs), model.predict(inputs), atol=0)
assert_allclose(clone(gbrt_best).fit(Xq_train,yq_train).predict(Xq_val),gbrt_best.predict(Xq_val))
assert_allclose(root_mean_squared_error(yq_val,regr.predict(Xq_val))**2, mean_squared_error(yq_val,regr.predict(Xq_val)))
print('All twenty original gist examples and three supplied missing examples passed.')
print('All 500 SAMME sample-weight stages and weighted votes match independent tree traversal.')
print('AdaBoost.R2 weighted medians and all 120 GBRT validation stages match independent calculations.')
print('All 600 XGBoost classifier/regressor rows match JSON-tree sums; model save/reload passes.')
print('Separate classification/regression data, RMSE squared = MSE, deterministic refit and row-order checks pass.')
thread_limits.restore_original_limits()

## Complete original gist snapshots (not executed)

### Original 1: 9f53f070f6d14915e49dfbd679ece2bf
Revision: `6e012a49daeceac255503d336073c93df01304d7`

```python
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons
import seaborn as sns 

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

plt.figure(figsize=(10,8))
sns.scatterplot(X[:, 0], X[:, 1],c=y)
plt.title('The moon dataset', fontsize=18)
plt.xlabel(r"X1", fontsize=18)
plt.ylabel(r"X2", fontsize=18, rotation=0)
```

### Original 2: bff7451e28212a46744e8f77df87aa51
Revision: `7ffe09a25c4a39c32e8574a721d5aa21f0831a33`

```python
from matplotlib.colors import ListedColormap

def plot_decision_boundary(clf, X, y, axes=[-1.5, 2.45, -1, 1.5], alpha=0.5, contour=True):
    x1s = np.linspace(axes[0], axes[1], 100)
    x2s = np.linspace(axes[2], axes[3], 100)
    x1, x2 = np.meshgrid(x1s, x2s)
    X_new = np.c_[x1.ravel(), x2.ravel()]
    y_pred = clf.predict(X_new).reshape(x1.shape)
    custom_cmap = ListedColormap(['#fafab0','#9898ff','#a0faa0'])
    plt.contourf(x1, x2, y_pred, alpha=0.3, cmap=custom_cmap)
    if contour:
        custom_cmap2 = ListedColormap(['#7d7d58','#4c4c7f','#507d50'])
        plt.contour(x1, x2, y_pred, cmap=custom_cmap2, alpha=0.8)
    plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo", alpha=alpha)
    plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs", alpha=alpha)
    plt.axis(axes)
    plt.xlabel(r"$x_1$", fontsize=18)
    plt.ylabel(r"$x_2$", fontsize=18, rotation=0)
    

plot_decision_boundary(ada_clf, X, y)
```

### Original 3: e69ad72dc57fd1c9c308e36c98934218
Revision: `2819b1dc92dd8c314f47b39d1143368cb778e330`

```python
np.random.seed(42)

X = np.random.rand(100, 1) - 0.5
y = 3*X[:, 0]**2 + 0.05 * np.random.randn(100)
plt.plot(X,y, '.')
plt.title('Quadratic Data')
plt.xlabel('X')
plt.ylabel('y')
```

### Original 4: 09cd57fc1636b5ab87b71c0ab4c47a21
Revision: `7ab4cf5f3c20c498e6bbd33cde7ff5acb4ea97e5`

```python
from sklearn.ensemble import AdaBoostRegressor
from sklearn.datasets import make_regression

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=49)
regr = AdaBoostRegressor(random_state=0, n_estimators=100)
regr.fit(X_train, y_train)
y_pred = regr.predict(X_val)
mean_squared_error(y_val, y_pred, squared=True)
```

### Original 5: ef273060e5624d55af133909f2ef41ad
Revision: `4edf5c32dd3c9229bc682d6a105c26c4aaeab18c`

```python
np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3*X[:, 0]**2 + 0.05 * np.random.randn(100)
plt.plot(X,y, '.')
plt.title('Quadratic Data')
plt.xlabel('X')
plt.ylabel('y')
```

### Original 6: a00242b2402fe3b740f9de163d0f53f3
Revision: `96dbfd1aa1e1f0c2e0da5553f74bc8676ad4349c`

```python
from sklearn.tree import DecisionTreeRegressor

tree_reg1 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg1.fit(X, y)
```

### Original 7: ee90445c6d6edd694876d8f077891d99
Revision: `a5d9a9b88ad9c1e19b2df42c5ab9c9acacb8d730`

```python
y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg2.fit(X, y2)
```

### Original 8: 0f96bc8e272e7d652801be9d4182420f
Revision: `2a6c4aeaa0a94d19bd1dcd1a28f06a0f0807fba6`

```python
y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg3.fit(X, y3)
```

### Original 9: cba1bc4f417dd2cf87f608d3503726f0
Revision: `28bec593ff4bba5f25f8c064a33c9d2e56f477c3`

```python
X_new = np.array([[0.8]])
y_pred = sum(tree.predict(X_new) for tree in (tree_reg1, tree_reg2, tree_reg3))
y_pred
```

### Original 10: 8e20a617577945c88cf613f016924bd4
Revision: `d8c05614e4692a27a9d54add0268ec6b152e3421`

```python
def plot_predictions(regressors, X, y, axes, label=None, style="r-", data_style="b.", data_label=None):
    x1 = np.linspace(axes[0], axes[1], 500)
    y_pred = sum(regressor.predict(x1.reshape(-1, 1)) for regressor in regressors)
    plt.plot(X[:, 0], y, data_style, label=data_label)
    plt.plot(x1, y_pred, style, linewidth=2, label=label)
    if label or data_label:
        plt.legend(loc="upper center", fontsize=16)
    plt.axis(axes)
```

### Original 11: 116f8e4c3bfa357c973a53767e138730
Revision: `be34d71ca15746fcf4a471d4f712247323c51353`

```python
plt.figure(figsize=(11,11))

plt.subplot(321)
plot_predictions([tree_reg1], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h_1(x_1)$", style="g-", data_label="Training set")
plt.ylabel("$y$", fontsize=16, rotation=0)
plt.title("Residuals and tree predictions", fontsize=16)

plt.subplot(322)
plot_predictions([tree_reg1], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1)$", data_label="Training set")
plt.ylabel("$y$", fontsize=16, rotation=0)
plt.title("Ensemble predictions", fontsize=16)

plt.subplot(323)
plot_predictions([tree_reg2], X, y2, axes=[-0.5, 0.5, -0.5, 0.5], label="$h_2(x_1)$", style="g-", data_style="k+", data_label="Residuals")
plt.ylabel("$y - h_1(x_1)$", fontsize=16)

plt.subplot(324)
plot_predictions([tree_reg1, tree_reg2], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1) + h_2(x_1)$")
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.subplot(325)
plot_predictions([tree_reg3], X, y3, axes=[-0.5, 0.5, -0.5, 0.5], label="$h_3(x_1)$", style="g-", data_style="k+")
plt.ylabel("$y - h_1(x_1) - h_2(x_1)$", fontsize=16)
plt.xlabel("$x_1$", fontsize=16)

plt.subplot(326)
plot_predictions([tree_reg1, tree_reg2, tree_reg3], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="$h(x_1) = h_1(x_1) + h_2(x_1) + h_3(x_1)$")
plt.xlabel("$x_1$", fontsize=16)
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.show()
```

### Original 12: 2e7c22371a275dba0ff895a71619cdb2
Revision: `3f3dbce25b5ace71d9dde2577d1da1c5e95823e0`

```python
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

gbclf = GradientBoostingClassifier(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbclf.fit(X_train, y_train)
y_pred = gbclf.predict(X_test)
print('Test accuarcy using Gradient Boosting Classifier', accuracy_score(y_test, y_pred))
```

### Original 13: 05db461def6b2ab9a6b7890660ec3628
Revision: `099284c7c0ca8ad7de5b18620a40a78e81722c25`

```python
from sklearn.ensemble import GradientBoostingRegressor

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbrt.fit(X, y)
```

### Original 14: 68642bbece9b022d812f3aa7502e7a74
Revision: `a6e129172ed9da781c65476b939201244ea1bee8`

```python
gbrt_slow = GradientBoostingRegressor(max_depth=2, n_estimators=200, learning_rate=0.1, random_state=42)
gbrt_slow.fit(X, y)
```

### Original 15: 07ccf155122eafa5cd2954c9dae2e802
Revision: `8a55a9a20738266dce36d27158ea2a8b39b78290`

```python
fix, axes = plt.subplots(ncols=2, figsize=(10,4), sharey=True)

plt.sca(axes[0])
plot_predictions([gbrt], X, y, axes=[-0.5, 0.5, -0.1, 0.8], label="Ensemble predictions")
plt.title("learning_rate={}, n_estimators={}".format(gbrt.learning_rate, gbrt.n_estimators), fontsize=14)
plt.xlabel("$x_1$", fontsize=16)
plt.ylabel("$y$", fontsize=16, rotation=0)

plt.sca(axes[1])
plot_predictions([gbrt_slow], X, y, axes=[-0.5, 0.5, -0.1, 0.8])
plt.title("learning_rate={}, n_estimators={}".format(gbrt_slow.learning_rate, gbrt_slow.n_estimators), fontsize=14)
plt.xlabel("$x_1$", fontsize=16)

plt.show()
```

### Original 16: 0d46cfec6b9270f7e67053952a9ee802
Revision: `6025ccf189f0221470cbc40d8b0c6b0d06f69f68`

```python
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=49)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120, random_state=42)
gbrt.fit(X_train, y_train)

errors = [mean_squared_error(y_val, y_pred)
          for y_pred in gbrt.staged_predict(X_val)]
bst_n_estimators = np.argmin(errors) + 1

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=bst_n_estimators, random_state=42)
gbrt_best.fit(X_train, y_train)
```

### Original 17: fa7382fa7afbe353dca0e44508987c17
Revision: `0beb5524252c24d499f348db27766753610c74c7`

```python
min_error = np.min(errors)
plt.figure(figsize=(15, 8))

plt.subplot(121)
plt.plot(np.arange(1, len(errors) + 1), errors, "b.-")
plt.plot([bst_n_estimators, bst_n_estimators], [0, min_error], "k--")
plt.plot([0, 120], [min_error, min_error], "k--")
plt.plot(bst_n_estimators, min_error, "ko")
plt.text(bst_n_estimators, min_error*1.2, "Minimum", ha="center", fontsize=14)
plt.axis([0, 120, 0, 0.01])
plt.xlabel("Number of trees")
plt.ylabel("Error", fontsize=16)
plt.title("Validation error", fontsize=14)

plt.subplot(122)a
plot_predictions([gbrt_best], X, y, axes=[-0.5, 0.5, -0.1, 0.8])
plt.title("Best model (%d trees)" % bst_n_estimators, fontsize=14)
plt.ylabel("$y$", fontsize=16, rotation=0)
plt.xlabel("$x_1$", fontsize=16)

plt.show()
```

### Original 18: 9264b0f7b07a1e96f8c5173231c07fc0
Revision: `6fa7d79a209ac1afbd7ba9dadf133aaaa4049b78`

```python
!pip install xgboost
```

### Original 19: f3ab4f26c20e8bf9cd59841cba90c5f8
Revision: `85a76ee0ce196c15591daea80920e946cb86651b`

```python
import xgboost as xgb
from sklearn.metrics import accuracy_score

# create the moon dataset
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


xgb_classifier = xgb.XGBClassifier()
xgb_classifier.fit(X_train,y_train)

predictions = xgb_classifier.predict(X_test)
print("classifcation accuracy of XGBoost :",accuracy_score(y_test,predictions))
```

### Original 20: b224b6c1c1b02dc76f6f3b5ca8fae4c2
Revision: `0753699790e87243c6eecc454015729abdc40ada`

```python
plot_decision_boundary(xgb_classifier, X_train, y_train, 'The decision bounradies of XGBoost')
```

